In [ ]:
import pandas as pd
import os
import s3fs


fs = s3fs.S3FileSystem(
    client_kwargs={'endpoint_url': 'https://'+'minio.lab.sspcloud.fr'},
    key = os.environ["AWS_ACCESS_KEY_ID"], 
    secret = os.environ["AWS_SECRET_ACCESS_KEY"], 
    token = os.environ["AWS_SESSION_TOKEN"])

s3_folder = "s3://lab/PESSD/csv"

files = fs.glob(f"{s3_folder}/*.csv")

for file in files:
    fs.get(file, "data/"+(file.removeprefix("lab/PESSD/csv/")))

In [8]:
df = pd.DataFrame()
for file in os.listdir("data") : 
    temp = pd.read_csv("data/"+file)
    df = pd.concat([df,temp])


In [21]:
BI = df[(df["audio_file"]=="BI_wav.wav") & ((df["start"]<(15*60))|((df["start"]>(17.5*60))&((df["stop"]<(89*60))))|(df["start"]>(92.5*60)))]

In [23]:
BIF = df[(df["audio_file"]=="BIF_wav.wav") & ((df["start"]<(12.5*60))|((df["start"]>(14.5*60))&((df["stop"]<(76*60)))))]

In [25]:
BMF = df[(df["audio_file"]=="BMF_wav.wav") & ((df["start"]>(1.5*60)))]

In [26]:
BMH = df[(df["audio_file"]=="BMH_wav.wav")]

In [28]:
SCR = df[(df["audio_file"]=="SCR2_wav.wav") & ((df["start"]<(28*60)))]

In [29]:
d = pd.concat([BI,BIF,BMH,BMF,SCR])

In [2]:
!pip install bertopic sentence-transformers umap-learn hdbscan

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 37.3 MB/s  0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 71.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 616.3/616.3 kB 88.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 86.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 57.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 80.7 MB/s  0:00:006m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.8/3.8 MB 31.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 72.0 MB/s  0:00:09m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 94.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 83.0 MB/s  0:00:06m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 99.9 MB/

In [3]:
import pandas as pd
d = pd.read_csv("data/test.csv")

In [ ]:
#Pas nécessaire
import spacy
nlp = spacy.load("fr_core_news_sm", disable=["parser", "ner"])
d['lemmatized_text'] = [" ".join([token.lemma_ for token in doc]) for doc in nlp.pipe(d['text'])]

In [7]:
groups = (d["main_g"] != d["main_g"].shift()).cumsum()

# merge consecutive texts
interventions = (
    d.groupby(groups)
      .agg({
          "main_g": "first",
          "text": " ".join
      })
      .reset_index(drop=True)
)

In [4]:
d

,Unnamed: 0,start,stop,text,main_speaker,main_g,audio_file
0,0,0.00,61.20,"En endroit, on est parti pour vivre encore un ...",SPEAKER_08,male,BI_wav.wav
1,1,61.20,103.60,"Oui, bonjour à tous, effectivement. Bienvenue ...",SPEAKER_02,male,BI_wav.wav
2,2,103.60,125.60,On imagine un Quentin qui aura envie de conser...,SPEAKER_12,male,BI_wav.wav
3,3,125.60,146.60,Moi je vais vous donner les numéros de Dossard...,SPEAKER_05,female,BI_wav.wav
4,4,146.60,150.60,Ouais l'équivalent à peu près de deux tours et...,SPEAKER_02,male,BI_wav.wav
...,...,...,...,...,...,...,...
1472,126,1611.68,1623.00,C'était raté sur la dernière course. La FP 4e ...,SPEAKER_11,male,SCR2_wav.wav
1473,127,1623.16,1633.56,"La Suède, qui revient à 3 secondes dès Nord-Vé...",SPEAKER_06,male,SCR2_wav.wav
1474,128,1634.88,1640.40,Ça va pas le mettre en confiance pour le début...,SPEAKER_10,male,SCR2_wav.wav
1475,129,1640.56,1645.64,C'était déjà la catastrophe à Nové-Mesto. Il a...,SPEAKER_06,female,SCR2_wav.wav


In [71]:
# Install dependencies if needed
# pip install bertopic sentence-transformers umap-learn hdbscan

from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer
from umap import UMAP

# Load CamemBERT embedding model
embedding_model = SentenceTransformer("dangvantuan/sentence-camembert-base")

STOPWORDS = [x.strip() for x in open('stopwords.txt').readlines()]

# Vectorizer for topic representation
vectorizer_model = CountVectorizer(
    stop_words=STOPWORDS,
    ngram_range=(1,2),
    min_df=2
)
umap_model = UMAP(n_neighbors=15, n_components=5, min_dist=0.0, random_state=42)
# Build BERTopic model
topic_model = BERTopic(
    embedding_model=embedding_model,
    vectorizer_model=vectorizer_model,
    umap_model=umap_model,
    language="french",
    calculate_probabilities=True,
    verbose=True
)

# Fit the model
topics, probs = topic_model.fit_transform(d2["text"])

# Show discovered topics
print(topic_model.get_topic_info())

# Show keywords for a specific topic
print(topic_model.get_topic(0))

# Visualize topics
topic_model.visualize_topics()

# Visualize topic hierarchy
topic_model.visualize_hierarchy()

#keybertopic, extrait phrases saillantes
#tester lda
#virer phrases nulles et refaire tourner
#pas virer -1 sauf si faible échantillon et checker
#checker si semi-superviser pour trouver thèmes

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4347.65it/s]
CamembertModel LOAD REPORT from: dangvantuan/sentence-camembert-base
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
2026-03-14 10:29:42,784 - BERTopic - Embedding - Transforming documents to embeddings.
Batches: 100%|██████████| 44/44 [11:50<00:00, 16.14s/it]
2026-03-14 10:41:33,081 - BERTopic - Embedding - Completed ✓
2026-03-14 10:41:33,082 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-03-14 10:41:37,388 - BERTopic - Dimensionality - Completed ✓
2026-03-14 10:41:37,389 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-03-14 10:41:37,453 - BERTopic - Cluster - Completed ✓
2026-03-14 10:41:37,457 - BERTopic - Representation - Fine-tuning topics using representation 

   Topic  Count                                 Name  \
0     -1     21  -1_ca tour_tour ca_norvégienne_tour   
1      0   1337             0_tir_20_course_vraiment   
2      1     20              1_20_20 20_joue_déjà 20   

                                      Representation  \
0  [ca tour, tour ca, norvégienne, tour, ca, cour...   
1  [tir, 20, course, vraiment, déjà, faute, derni...   
2  [20, 20 20, joue, déjà 20, 20 19, 19, hier, jo...   

                                 Representative_Docs  
0  [Ca va faire un beau dernier tour., Ca va fair...  
1  [C'est Téline Grossiana qui est en train d'exp...  
2  [Il joue le 20 sur 20., Il joue le 20 sur 20.,...  
[('tir', np.float64(0.1724916559983226)), ('20', np.float64(0.142516035526065)), ('course', np.float64(0.13518981414223927)), ('vraiment', np.float64(0.10525233700419431)), ('déjà', np.float64(0.09727076113771353)), ('faute', np.float64(0.0744283502135292)), ('dernière', np.float64(0.0744283502135292)), ('15', np.float64(0.07159

ValueError: zero-size array to reduction operation maximum which has no identity

In [72]:
t = pd.DataFrame({
    "text": d2['text'],
    "topic": topics,
    "topic_prob": probs.max(axis=1)  # max probability per text
})

In [73]:
t["topic"].value_counts()

topic
 0    1337
-1      21
 1      20
Name: count, dtype: int64

In [67]:
topic_model.get_topic(4,full=True)

False

In [ ]:
#1 : 4, 5, 14, 18, 20 retirés
for k in t[t["topic"]==22]["text"] : 
    print(k)

In [58]:
mask = t[(t["topic"]!=4)&(t["topic"]!=5)&(t["topic"]!=14)&(t["topic"]!=18)&(t["topic"]!=20)]["text"]

In [60]:
d2 = d.merge(mask, on="text")

In [48]:
topic_model.get_topic(-1)

[('tir', np.float64(0.019705384158766548)),
 ('course', np.float64(0.01810100205230608)),
 ('julia', np.float64(0.016166922702569482)),
 ('secondes', np.float64(0.01581559885555621)),
 ('petit', np.float64(0.015216028128888991)),
 ('vraiment', np.float64(0.014750896397992897)),
 ('faut', np.float64(0.014264980364740047)),
 ('allez', np.float64(0.014103260774615873)),
 ('20', np.float64(0.01389915649281138)),
 ('aller', np.float64(0.013604787866193726))]

In [65]:
test = topic.merge(d, on="text")

In [67]:
test[(test["topic"]==10)|(test["topic"]==15)]

,text,topic,topic_prob,start,stop,main_speaker,main_g,audio_file
8,Pour vous en dire un petit peu plus sur la pis...,10,1.000000,289.60,345.80,SPEAKER_05,female,BI_wav.wav
9,On discutait hier justement dans la reconnaiss...,10,0.124827,345.80,356.40,SPEAKER_12,male,BI_wav.wav
11,Après c'est vrai que cette année il y a des st...,15,1.000000,374.20,386.60,SPEAKER_05,female,BI_wav.wav
13,"C'est un athlète solide, le jeune Suisse qui p...",15,0.238053,421.20,434.20,SPEAKER_12,male,BI_wav.wav
25,C'est un athlète qui a énormément d'expérience...,15,0.219121,604.20,622.20,SPEAKER_12,male,BI_wav.wav
26,"Cette année, il est un tout petit peu moins bo...",15,1.000000,622.20,634.20,SPEAKER_05,female,BI_wav.wav
28,"Fabien, c'est vraiment un skieur qui fait part...",15,1.000000,677.20,709.20,SPEAKER_05,female,BI_wav.wav
78,"Là, ça fait mal. On voit dans le regard qu'il ...",10,0.364798,1586.20,1595.20,SPEAKER_12,female,BI_wav.wav
93,On a l'impression qu'il y a les jambes qui tre...,10,1.000000,1724.20,1734.20,SPEAKER_05,female,BI_wav.wav
639,"Comme on l'avait dit pour le relais mixte, c'e...",10,0.152644,4025.84,4036.72,SPEAKER_05,female,BI_wav.wav
